# CascadeDebug — Phase 7 GRPO (Colab) — **upload this file directly**

1. In Colab: **File → Upload notebook** and choose this `.ipynb`, **or** upload the whole `colab_phase7` folder and open this file.
2. **Runtime → Change runtime type → T4 GPU** (Pro: A100 is faster).
3. Run all cells top to bottom. Training reads code from the GitHub repo (clone + `main`), not from a local copy of the `.py` alone.

**What you get by default** (`train_grpo_colab.py` on `main`, `PROFILE = "submission"`):

| Item | Value |
|------|--------|
| Model | Qwen2.5-**3B** 4-bit (Unsloth) |
| Steps | **90** (not 300) |
| Max completion | **160** tokens |
| Grad accum | 4 |
| Time | typically **1–2.5 h** on T4 (under ~3 h target) |

`PROFILE = "full"` in the same script = **7B** + **300** steps (often **5 h+** on T4) — use only if you are not on a tight deadline.

**After training:** download `results/*.png`, `results/training_log.csv`, and optionally `results/lora_adapter/`.


In [ ]:
# GPU check
!nvidia-smi

In [ ]:
# Install (mergekit is required — TRL's GRPOTrainer imports it; ~1–2 min)
%pip install -q unsloth "trl>=0.12.0" mergekit datasets matplotlib numpy huggingface_hub
print("OK: dependencies installed.")

In [ ]:
import os
from getpass import getpass

# Optional: Hugging Face write token for uploading plots to your Space (repo results/)
tok = getpass("HF write token (or press Enter to skip): ").strip()
if tok:
    os.environ["HF_TOKEN"] = tok
    print("HF_TOKEN set (length:", len(tok), ")")
else:
    print("No HF token — results stay local under results/ only.")

In [ ]:
# Clone and sync to latest main (so `train_grpo_colab.py` is current — submission = 3B, 90 steps)
import os, subprocess
REPO = "/content/cascadedebug"
CLONE = "https://github.com/sparshagra/cascadedebug.git"
if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "clone", CLONE, REPO], check=True)
subprocess.run("git fetch origin && git reset --hard origin/main", shell=True, cwd=REPO, check=True)
subprocess.run(["git", "log", "-1", "--oneline"], cwd=REPO)
os.chdir(REPO)  # so %run and paths resolve
print("CWD set to", REPO)

In [ ]:
# Quick check: fast profile must be present in the file pulled from GitHub
f = "/content/cascadedebug/training/train_grpo_colab.py"
t = open(f, encoding="utf-8").read()
assert 'PROFILE = "submission"' in t, "Set PROFILE in script or pull latest main from GitHub"
assert "MAX_STEPS = 90" in t, "Outdated script — re-run the clone/reset cell (expect 90 steps, not 300)"
assert "Qwen2.5-3B" in t and "MAX_COMPLETION_LENGTH = 160" in t, "Check train_grpo_colab.py submission branch on main"
print("OK: submission profile (3B, 90 steps, 160) detected — you can start training in the next cell.")

In [ ]:
# Train (re-installs nothing; uses repo you just synced)
import os
os.chdir("/content/cascadedebug")
%run training/train_grpo_colab.py